# AP Commander â€” Baseline Eval + GRPO Training

**Run on:** Google Colab (T4/A100) Â· HF Spaces (A10G) Â· any CUDA machine  
**Environment:** `https://pathikreet-ap-clerk-env.hf.space` (always live)  
**Repo:** `Pathikreet/ap-commander-training`

```
This notebook has two independent sections:
  Part A â€” Baseline Eval   (no training, measure untrained model)
  Part B â€” GRPO Training   (fine-tune with curriculum + accumulated rewards)
```

Run Part A first to get the baseline, then Part B to train. Each run saves to a
timestamped folder under `runs/` so nothing is ever overwritten.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU â€” go to Runtime > Change runtime type > GPU (T4 or A100)'
gpu = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'GPU : {gpu.name}')
print(f'VRAM: {vram:.1f} GB')
if vram < 14:
    print('WARNING: < 14 GB VRAM â€” use Qwen2.5-1.5B or reduce batch size')
else:
    print('OK: enough VRAM for Qwen2.5-7B in 4-bit')

In [ ]:
# Install â€” TRL standard stack (no Unsloth: avoids llm_blender conflicts on HF Spaces)
!pip install -q 'trl>=0.15.0' accelerate peft transformers bitsandbytes
!pip install -q requests datasets matplotlib huggingface_hub
print('Done')

In [ ]:
import os

# Reduce CUDA memory fragmentation â€” must be set before torch is imported
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# â”€â”€ CONFIG â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MODEL_NAME      = 'Qwen/Qwen2.5-1.5B-Instruct'  # fast, fits any GPU
# MODEL_NAME    = 'Qwen/Qwen2.5-7B-Instruct'    # stronger, needs A100/A10G
# MODEL_NAME    = 'meta-llama/Meta-Llama-3-8B-Instruct'  # needs HF_TOKEN

HF_TOKEN        = ''   # only for gated models (Llama etc.); Qwen is public â€” leave blank
NUM_EPOCHS      = 3
NUM_GENERATIONS = 8    # GRPO group size: 8 for T4, 16 for A100/A10G
EVAL_SEEDS      = [42, 99, 7]
ENV_URL         = 'https://pathikreet-ap-clerk-env.hf.space'
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('[AUTH] Logged in â€” gated models available')
else:
    print('[AUTH] No HF_TOKEN â€” public models only (Qwen works without token)')

print(f'Model : {MODEL_NAME}')
print(f'Epochs: {NUM_EPOCHS}  |  Generations: {NUM_GENERATIONS}')
print(f'CUDA alloc: {os.environ.get("PYTORCH_CUDA_ALLOC_CONF")}')


In [ ]:
import requests

h = requests.get(f'{ENV_URL}/health', timeout=30).json()
print(f"Environment: {h['status']}  |  tasks: {h.get('total_tasks')}  |  version: {h.get('version')}")

# Quick sanity: one episode
reset = requests.post(f'{ENV_URL}/reset', json={'task_id': 'easy_perfect_match', 'seed': 42}).json()
step  = requests.post(f'{ENV_URL}/step', json={
    'session_id': reset['session_id'],
    'action': {'decision': 'APPROVE_FULL',
               'approved_amount': reset['observation']['invoice']['invoice_total'],
               'reason_code': 'MATCH_CONFIRMED',
               'explanation': 'Invoice matches PO and GRN. Three-way match confirmed.'}
}).json()
print(f"Sanity check: score={step['reward']['score']}  (expect ~0.99)")

In [ ]:
import json, re, random, time, datetime, collections, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import requests

SYSTEM_PROMPT = (
    "You are an AI Accounts Payable Clerk. Review the invoice, PO, and GRN, "
    "then output ONLY a single valid JSON object. No prose, no markdown, no explanation outside the JSON.\n\n"
    "Valid decisions: APPROVE_FULL | APPROVE_PARTIAL | REJECT | ESCALATE | QUERY_VENDOR\n"
    "Valid reason codes: MATCH_CONFIRMED | QUANTITY_MISMATCH | PRICE_DISCREPANCY | POLICY_VIOLATION | "
    "NO_PO_FOUND | DUPLICATE_INVOICE | VENDOR_MISMATCH | TAX_DISCREPANCY | PENDING_CLARIFICATION | MANAGER_REVIEW\n\n"
    'Example (generate your own â€” do not copy):\n'
    '{"decision": "REJECT", "approved_amount": 0.0, "reason_code": "NO_PO_FOUND", '
    '"explanation": "Invoice INV-2024-5821 rejected: no open PO found for vendor TechCorp. Policy Rule 5 mandates a valid OPEN PO."}\n\n'
    "Your response must start with { and end with } with no other text."
)

VALID_DECISIONS    = {'APPROVE_FULL','APPROVE_PARTIAL','REJECT','ESCALATE','QUERY_VENDOR','HOLD'}
VALID_REASON_CODES = {'MATCH_CONFIRMED','QUANTITY_MISMATCH','PRICE_DISCREPANCY','POLICY_VIOLATION',
                      'NO_PO_FOUND','DUPLICATE_INVOICE','VENDOR_MISMATCH','TAX_DISCREPANCY',
                      'PENDING_CLARIFICATION','MANAGER_REVIEW'}

# All 20 tasks: 2 easy, 4 medium, 7 hard, 7 long-horizon
EVAL_TASKS = TRAIN_TASKS = [
    'easy_perfect_match',        'easy_no_po_found',
    'medium_quantity_shortfall', 'medium_price_discrepancy',
    'medium_split_delivery',     'medium_vendor_mismatch',
    'hard_policy_violation',     'hard_duplicate_invoice',
    'hard_partial_po_match',     'hard_tax_discrepancy',
    'hard_currency_conversion',  'hard_manager_preapproval', 'hard_credit_memo',
    'long_invoice_dispute',      'long_policy_migration',
    'long_batch_reconciliation', 'long_manager_chain',
    'long_fraud_investigation',  'long_audit_trail',
    'long_multi_vendor_split',
]

TASK_DIFFICULTY = {
    'easy_perfect_match': 'easy',        'easy_no_po_found': 'easy',
    'medium_quantity_shortfall': 'medium','medium_price_discrepancy': 'medium',
    'medium_split_delivery': 'medium',    'medium_vendor_mismatch': 'medium',
    'hard_policy_violation': 'hard',      'hard_duplicate_invoice': 'hard',
    'hard_partial_po_match': 'hard',      'hard_tax_discrepancy': 'hard',
    'hard_currency_conversion': 'hard',   'hard_manager_preapproval': 'hard', 'hard_credit_memo': 'hard',
    'long_invoice_dispute': 'long',       'long_policy_migration': 'long',
    'long_batch_reconciliation': 'long',  'long_manager_chain': 'long',
    'long_fraud_investigation': 'long',   'long_audit_trail': 'long',
    'long_multi_vendor_split': 'long',
}
DIFF_COLORS = {'easy': '#3fb950', 'medium': '#d29922', 'hard': '#f85149', 'long': '#a371f7'}
DIFF_ORDER  = ['easy', 'medium', 'hard', 'long']


def obs_to_prompt(obs):
    inv   = obs['invoice']
    lines = '\n'.join(f"  {li['description']}: qty={li['quantity']}, unit_price=${li['unit_price']:.2f}"
                      for li in inv.get('line_items', []))
    pos   = '\n'.join(
        f"  PO {p['po_number']} ({p['status']}) {p['vendor_name']}: " +
        ', '.join(f"{l['description']} qty={l['ordered_quantity']} @${l['agreed_unit_price']:.2f}"
                  for l in p.get('lines', []))
        for p in obs.get('purchase_orders', []))
    grns  = '\n'.join(
        f"  GRN {g['grn_id']}: " + ', '.join(f"{l['description']} recv={l['received_quantity']}"
                                              for l in g.get('lines', []))
        for g in obs.get('goods_receipts', []))
    ctx   = '\n'.join(f'  {n}' for n in obs.get('context_notes', []))
    paid  = ', '.join(obs.get('paid_invoice_ids', []))
    return (
        f"TASK: {obs['task_name']}\n{obs['task_description']}\n\n"
        f"INVOICE {inv['invoice_id']} | {inv['vendor_name']} | ${inv['invoice_total']:,.2f}\n{lines}\n"
        f"Freight: ${inv.get('freight_charge',0):.2f}\n\n"
        f"PURCHASE ORDERS:\n{pos}\n\nGOODS RECEIPTS:\n{grns}\n"
        + (f"PAID LEDGER: {paid}\n" if paid else "")
        + (f"CONTEXT:\n{ctx}\n"     if ctx  else "")
        + f"\nPOLICY:\n{obs['company_policy']}\n\nOutput JSON decision."
    )


def parse_action(raw):
    clean = re.sub(r'```(?:json)?\s*|\s*```', '', raw).strip()
    m = re.search(r'\{.*\}', clean, re.DOTALL)
    if m:
        try:
            a = json.loads(m.group())
            if (a.get('decision') in VALID_DECISIONS and
                a.get('reason_code') in VALID_REASON_CODES and
                isinstance(a.get('approved_amount'), (int, float)) and
                len(a.get('explanation', '')) > 10):
                return a, True
        except Exception:
            pass
    return {'decision': 'REJECT', 'approved_amount': 0.0,
            'reason_code': 'NO_PO_FOUND', 'explanation': 'parse error'}, False


def _greedy_followup(obs_dict):
    notes = ' '.join(obs_dict.get('context_notes', [])).lower()
    total = abs(float(obs_dict.get('invoice', {}).get('invoice_total', 0) or 0))
    if any(k in notes for k in ('manager approved', 'vp approved', 'cfo approved',
                                 'pre-approved', 'pre-approv', 'approved by')):
        return {'decision': 'APPROVE_FULL', 'approved_amount': total,
                'reason_code': 'MATCH_CONFIRMED',
                'explanation': f'Approval confirmed via escalation. Approving ${total:.2f}.'}
    if 'compliance' in notes and any(k in notes for k in ('cleared', 'approved', 'pass')):
        return {'decision': 'APPROVE_FULL', 'approved_amount': total,
                'reason_code': 'MATCH_CONFIRMED',
                'explanation': f'Compliance cleared. Approving ${total:.2f}.'}
    if any(k in notes for k in ('fraudulent', 'duplicate', 'already paid', 'deny', 'invalid')):
        return {'decision': 'REJECT', 'approved_amount': 0.0,
                'reason_code': 'DUPLICATE_INVOICE',
                'explanation': 'Confirmed fraud/duplicate. Rejecting.'}
    if any(k in notes for k in ('flagged', 'violation', 'sox', 'gdpr', 'non-compliant')):
        return {'decision': 'REJECT', 'approved_amount': 0.0,
                'reason_code': 'POLICY_VIOLATION',
                'explanation': 'Compliance violation confirmed. Rejecting.'}
    if any(k in notes for k in ('confused', 'unclear', 'unable to confirm')):
        return {'decision': 'ESCALATE', 'approved_amount': 0.0,
                'reason_code': 'MANAGER_REVIEW',
                'explanation': 'Vendor response ambiguous. Escalating.'}
    return {'decision': 'REJECT', 'approved_amount': 0.0,
            'reason_code': 'PENDING_CLARIFICATION',
            'explanation': 'Could not resolve after investigation. Rejecting for safety.'}


def run_episode_accumulated(task_id, first_action, seed=None, discount=0.9, max_steps=20):
    # Returns (score: float, episode_length: int)
    # QUERY_VENDOR->REJECT = 0.01 + 0.9*0.99 = 0.901 > shortcut REJECT ~0.4
    try:
        r = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id, 'seed': seed}, timeout=20)
        r.raise_for_status()
        session_id = r.json()['session_id']
        action, total, steps_taken = first_action, 0.0, 0
        for step_n in range(max_steps):
            result = requests.post(f'{ENV_URL}/step',
                                   json={'session_id': session_id, 'action': action},
                                   timeout=20)
            result.raise_for_status()
            result = result.json()
            total += (discount ** step_n) * float(result['reward']['score'])
            steps_taken = step_n + 1
            if result['done']:
                break
            action = _greedy_followup(result['observation'])
        return min(0.99, max(0.01, total)), steps_taken
    except Exception:
        return 0.01, 1


# â”€â”€ Dark-theme plot helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BG, PANEL, GRID = '#0d1117', '#161b22', '#21262d'
TXT, DIM        = '#e6edf3', '#8b949e'

def dark_fig(*args, **kwargs):
    fig = plt.figure(*args, **kwargs)
    fig.patch.set_facecolor(BG)
    return fig

def style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors='#c9d1d9', labelsize=8)
    for sp in ax.spines.values(): sp.set_color('#30363d')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=0.7); ax.set_axisbelow(True)
    if title:  ax.set_title(title,  color=TXT, fontsize=10, fontweight='bold', pad=8)
    if xlabel: ax.set_xlabel(xlabel, color=DIM, fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, color=DIM, fontsize=8)

baseline_results = {}   # populated by Part A; guarded in Part B cells
print('Helper functions ready.')
diff_counts = {d: sum(1 for v in TASK_DIFFICULTY.values() if v == d) for d in DIFF_ORDER}
print('Tasks: ' + str(len(EVAL_TASKS)) + ' â€” ' + '  '.join(f'{d}:{diff_counts[d]}' for d in DIFF_ORDER))


---
## Part A â€” Untrained Baseline Evaluation
Loads the model **without any fine-tuning**, evaluates all 10 tasks Ã— 3 seeds,
produces plots, and saves to `runs/baselines/MODEL-DATETIME/`.

**Skip to Part B** if you only want to train.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
model_base.eval()
print(f'Loaded {MODEL_NAME} (4-bit NF4, no LoRA)')

In [ ]:
def eval_one(model, task_id, seed):
    try:
        reset = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id, 'seed': seed}, timeout=20).json()
        obs, sid = reset['observation'], reset['session_id']
        msgs = [{'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':obs_to_prompt(obs)}]
        text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=250, temperature=0.1, do_sample=True)
        raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action, fmt_ok = parse_action(raw)
        score = float(requests.post(f'{ENV_URL}/step',
                                    json={'session_id': sid, 'action': action},
                                    timeout=20).json()['reward']['score'])
        return score, raw[:120], action.get('decision','?'), fmt_ok
    except Exception as e:
        return 0.01, str(e), 'ERROR', False


print(f'Evaluating {len(EVAL_TASKS)} tasks Ã— {len(EVAL_SEEDS)} seeds = {len(EVAL_TASKS)*len(EVAL_SEEDS)} episodes\n')
baseline_results = {}
parse_failures   = 0

for task_id in EVAL_TASKS:
    diff = TASK_DIFFICULTY[task_id]
    scores, decisions, fmts = [], [], []
    for seed in EVAL_SEEDS:
        score, raw, dec, fmt_ok = eval_one(model_base, task_id, seed)
        scores.append(score); decisions.append(dec); fmts.append(fmt_ok)
        if not fmt_ok: parse_failures += 1
        print(f'  [{diff[:4]}] {task_id:<35} seed={seed}  score={score:.3f}  {dec}  fmt={fmt_ok}')
        print(f'    {raw[:90]}')
        time.sleep(0.2)
    baseline_results[task_id] = {
        'difficulty': diff,
        'scores':     [round(s,4) for s in scores],
        'mean':       round(sum(scores)/len(scores), 4),
        'decisions':  decisions,
        'fmt_rate':   round(sum(fmts)/len(fmts), 3),
    }

by_diff = {}
for tid, v in baseline_results.items():
    by_diff.setdefault(v['difficulty'], []).append(v['mean'])

overall_baseline = sum(v['mean'] for v in baseline_results.values()) / len(baseline_results)
print(f'\nOverall mean: {overall_baseline:.3f}  parse_failures: {parse_failures}/{len(EVAL_TASKS)*len(EVAL_SEEDS)}')
for d in ['easy','medium','hard']:
    ms = by_diff.get(d,[])
    if ms: print(f'  {d}: {sum(ms)/len(ms):.3f}')

In [ ]:
fig = dark_fig(figsize=(16, 6))
gs  = fig.add_gridspec(1, 2, wspace=0.32)

# Panel 1 â€” per-task horizontal bars
ax1    = fig.add_subplot(gs[0,0])
tasks  = list(baseline_results.keys())
means  = [baseline_results[t]['mean'] for t in tasks]
colors = [DIFF_COLORS[baseline_results[t]['difficulty']] for t in tasks]
short  = [t.replace('easy_','').replace('medium_','').replace('hard_','').replace('_',' ').title()
          for t in tasks]
yp = range(len(tasks))
ax1.barh(list(yp), means, color=colors, alpha=0.85, edgecolor=BG)
ax1.set_yticks(list(yp)); ax1.set_yticklabels(short, fontsize=8)
ax1.set_xlim(0, 1.05)
ax1.axvline(overall_baseline, color='#f78166', linestyle='--', linewidth=1.2,
            label=f'Mean {overall_baseline:.3f}')
ax1.axvline(0.5, color='#484f58', linestyle=':', linewidth=1)
for i,m in enumerate(means):
    ax1.text(m+0.01, i, f'{m:.3f}', va='center', color='#c9d1d9', fontsize=8)
from matplotlib.patches import Patch
leg = [Patch(facecolor=c, label=d) for d,c in DIFF_COLORS.items()]
leg.append(plt.Line2D([0],[0], color='#f78166', linestyle='--', label=f'Mean {overall_baseline:.3f}'))
ax1.legend(handles=leg, fontsize=8, facecolor=PANEL, edgecolor='#30363d',
           labelcolor='#c9d1d9', loc='lower right')
style_ax(ax1, f'Per-Task Baseline ({len(EVAL_SEEDS)} seeds)', ylabel='Score')

# Panel 2 â€” mean by difficulty
ax2    = fig.add_subplot(gs[0,1])
diffs  = ['easy','medium','hard']
dmeans = [sum(by_diff.get(d,[0]))/max(1,len(by_diff.get(d,[0]))) for d in diffs]
bars   = ax2.bar(diffs, dmeans, color=[DIFF_COLORS[d] for d in diffs],
                 alpha=0.85, edgecolor=BG, width=0.5)
for i,(d,m) in enumerate(zip(diffs,dmeans)):
    ax2.text(i, m+0.02, f'{m:.3f}', ha='center', color='#c9d1d9', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.axhline(overall_baseline, color='#f78166', linestyle='--', linewidth=1,
            label=f'Overall {overall_baseline:.3f}')
ax2.legend(fontsize=8, facecolor=PANEL, edgecolor='#30363d', labelcolor='#c9d1d9')
style_ax(ax2, 'Mean by Difficulty', ylabel='Mean Score')

model_short = MODEL_NAME.split('/')[-1]
fig.suptitle(f'{model_short} â€” Untrained Baseline  |  4-bit NF4  |  {len(EVAL_SEEDS)} seeds  |  '
             f'overall={overall_baseline:.3f}  |  {datetime.datetime.now().strftime("%Y-%m-%d")}',
             color=TXT, fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/baseline_plot.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print('Plot saved: /tmp/baseline_plot.png')

In [ ]:
import shutil

model_slug = MODEL_NAME.split('/')[-1].lower().replace('.', '-')
ts         = datetime.datetime.now().strftime('%Y-%m-%d_%H%M')
run_dir    = f'/tmp/runs/baselines/{model_slug}-{ts}'
os.makedirs(run_dir, exist_ok=True)

output = {
    'run_type':       'llm_baseline_no_finetuning',
    'model':          MODEL_NAME,
    'quantization':   '4-bit NF4',
    'lora':           None,
    'timestamp':      datetime.datetime.now().isoformat(),
    'env_url':        ENV_URL,
    'seeds':          EVAL_SEEDS,
    'eval_tasks':     EVAL_TASKS,
    'overall_mean':   round(overall_baseline, 4),
    'parse_failures': parse_failures,
    'by_difficulty':  {d: round(sum(ms)/len(ms),4) for d,ms in by_diff.items()},
    'tasks':          baseline_results,
}
json_path = os.path.join(run_dir, 'baseline_results.json')
with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)
shutil.copy('/tmp/baseline_plot.png', os.path.join(run_dir, 'baseline_plot.png'))

# Optional: mount Google Drive and copy there for persistence across sessions
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copytree(run_dir, f'/content/drive/MyDrive/ap_commander/{os.path.basename(run_dir)}')

print(f'Saved to {run_dir}')
print(f'  baseline_results.json')
print(f'  baseline_plot.png')
print('Tip: use the cell below to download files, or mount Google Drive above.')

---
## Part B â€” GRPO Training

Adds LoRA to the loaded model (or reloads fresh), builds a curriculum-weighted
dataset, trains with two independent reward functions, and produces the full
4-panel results figure. Run Part A first to get `model_base` and `baseline_results`.

| Setting | Value | Why |
|---|---|---|
| `per_device_train_batch_size = num_generations` | 8 | TRL divisibility requirement |
| Two reward functions | env + format | Separate signals per guide |
| `run_episode_accumulated()` | discount=0.9 | QUERY_VENDORâ†’REJECT = 0.901 > shortcut REJECT 0.4 |
| `CurriculumSampler` | easyâ†’mediumâ†’hard | Unlocks harder tasks as easy mean â‰¥ 0.70 |

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

model_base.train()
model_base.enable_input_require_grads()
model_base.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,   # lora_alpha=32 matches train.py
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model_base, lora_cfg)
model.print_trainable_parameters()


In [ ]:
_DIFFICULTY_ORDER  = ['easy', 'medium', 'hard', 'long']
_UNLOCK_THRESHOLDS = {'easy': 0.70, 'medium': 0.65, 'hard': 0.60}

class CurriculumSampler:
    def __init__(self):
        self._rewards = collections.defaultdict(list)
        self.unlocked = {'easy'}
    def record(self, tid, r):
        self._rewards[tid].append(r)
        self._try_unlock()
    def mean_for(self, diff):
        v = [r for tid, d in TASK_DIFFICULTY.items()
             if d == diff for r in self._rewards.get(tid, [])]
        return sum(v) / len(v) if v else 0.0
    def _try_unlock(self):
        for i, d in enumerate(_DIFFICULTY_ORDER[:-1]):
            if d in self.unlocked and self.mean_for(d) >= _UNLOCK_THRESHOLDS.get(d, 0.70):
                nxt = _DIFFICULTY_ORDER[i + 1]
                if nxt not in self.unlocked:
                    self.unlocked.add(nxt)
                    print(f'[CURRICULUM] Unlocked {nxt}! mean({d})={self.mean_for(d):.3f}')
    def gate_task(self, tid):
        if TASK_DIFFICULTY.get(tid, 'easy') in self.unlocked:
            return tid
        return random.choice([t for t, d in TASK_DIFFICULTY.items() if d == 'easy'])
    def status(self):
        return ' | '.join(
            f'{d}={self.mean_for(d):.2f}{"+" if d in self.unlocked else "-"}'
            for d in _DIFFICULTY_ORDER)


class Metrics:
    def __init__(self):
        self.step            = 0
        self.reward_history  = []
        self.decision_counts = collections.Counter()
        self.format_scores   = []
        self.reward_by_task  = collections.defaultdict(list)
        self.parse_failures  = 0
        self.total_calls     = 0
        self._t0             = time.time()
    def log(self, rewards, decisions, fmt_oks, task_ids):
        self.step        += 1
        self.total_calls += len(rewards)
        self.reward_history.append((self.step, sum(rewards) / len(rewards)))
        for d in decisions: self.decision_counts[d] += 1
        for ok in fmt_oks:  self.format_scores.append(1.0 if ok else 0.0)
        for tid, r in zip(task_ids, rewards): self.reward_by_task[tid].append(r)
    def recent_mean(self, n=20):
        tail = self.reward_history[-n:]
        return sum(r for _, r in tail) / len(tail) if tail else 0.0
    def fmt_rate(self):
        return sum(self.format_scores) / len(self.format_scores) if self.format_scores else 0.0
    def summary(self):
        task_means = {t: round(sum(v)/len(v), 3) for t, v in self.reward_by_task.items()}
        elapsed    = (time.time() - self._t0) / 60
        print(f'[METRICS] step={self.step} recent={self.recent_mean():.3f} '
              f'fmt={self.fmt_rate():.1%} parse_fails={self.parse_failures} '
              f'calls={self.total_calls} elapsed={elapsed:.1f}min')
        print(f'[METRICS] per_task: {task_means}')
        print(f'[CURRICULUM] {CURRICULUM.status()}')

CURRICULUM = CurriculumSampler()
METRICS    = Metrics()
print('4-level curriculum (easy -> medium -> hard -> long) and Metrics ready.')
print('Currently unlocked:', CURRICULUM.unlocked)


In [ ]:
LOG_EVERY = 20

def env_reward_fn(completions, task_id=None, seed=None, **kwargs):
    # Environment reward: accumulated discounted per-step reward from AP Commander.
    task_ids = task_id if task_id is not None else ['easy_perfect_match'] * len(completions)
    seeds    = seed    if seed    is not None else [random.randint(1, 999)] * len(completions)
    rewards, decisions, fmts = [], [], []
    for completion, tid, s in zip(completions, task_ids, seeds):
        gated          = CURRICULUM.gate_task(tid)
        action, fmt_ok = parse_action(completion)
        score, ep_len  = run_episode_accumulated(gated, action, seed=int(s))
        rewards.append(score)
        decisions.append(action.get('decision', '?'))
        fmts.append(fmt_ok)
        CURRICULUM.record(gated, score)
        if not fmt_ok:
            METRICS.parse_failures += 1
        if METRICS.total_calls % LOG_EVERY == 0:
            print(f'  [sample] {gated} s={s} score={score:.3f} ep_len={ep_len} '
                  f'{action.get("decision")} fmt={fmt_ok}')
            print(f'  curriculum: {CURRICULUM.status()}')
    METRICS.log(rewards, decisions, fmts, list(task_ids))
    if METRICS.step % 5 == 0:
        METRICS.summary()
    return rewards

def format_reward_fn(completions, **kwargs):
    # Format reward: +0.15 valid JSON / -0.15 invalid
    return [0.15 if parse_action(c)[1] else -0.15 for c in completions]

# Smoke tests
t = env_reward_fn(
    ['{"decision":"APPROVE_FULL","approved_amount":100.0,"reason_code":"MATCH_CONFIRMED","explanation":"Invoice $100.00 matches PO and GRN exactly."}'],
    task_id=['easy_perfect_match'], seed=[42]
)
print(f'env_reward smoke: {t[0]:.3f}  (expect > 0.5)')
print(f'fmt_reward valid: {format_reward_fn(["{}"])[0]}')
print(f'fmt_reward bad:   {format_reward_fn(["bad json"])[0]}  (expect -0.15)')


In [ ]:
from datasets import Dataset

# More seeds for harder tasks â€” more variation helps discover multi-step sequences
_SEEDS_PER_DIFF = {'easy': 3, 'medium': 5, 'hard': 10, 'long': 10}
task_seed_pairs = [
    (tid, s)
    for tid in TRAIN_TASKS
    for s in range(1, _SEEDS_PER_DIFF[TASK_DIFFICULTY[tid]] + 1)
]
by_d = {d: sum(1 for tid, _ in task_seed_pairs if TASK_DIFFICULTY[tid] == d) for d in DIFF_ORDER}
print(f'[DATASET] {len(task_seed_pairs)} prompts: ' +
      '  '.join(f'{d}:{by_d[d]}' for d in DIFF_ORDER))
print('[DATASET] Building prompts via HTTP...')

rows = []
for task_id, seed in task_seed_pairs:
    try:
        reset = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id, 'seed': seed}, timeout=20).json()
        msgs  = [{'role': 'system', 'content': SYSTEM_PROMPT},
                 {'role': 'user',   'content': obs_to_prompt(reset['observation'])}]
        rows.append({
            'prompt':  tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
            'task_id': task_id,
            'seed':    seed,
        })
    except Exception as e:
        print(f'  skip {task_id} seed={seed}: {e}')

dataset = Dataset.from_list(rows)
by_d2 = {d: sum(1 for r in rows if TASK_DIFFICULTY[r['task_id']] == d) for d in DIFF_ORDER}
print(f'[DATASET] {len(dataset)} prompts loaded: ' + '  '.join(f'{d}:{by_d2[d]}' for d in DIFF_ORDER))
print(f'[CURRICULUM] {CURRICULUM.status()}')


In [ ]:
from trl import GRPOConfig, GRPOTrainer
model.train()

# per_device_train_batch_size must equal num_generations (TRL GRPO requirement)
config = GRPOConfig(
    output_dir                  = '/tmp/ap_commander_grpo',
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = NUM_GENERATIONS,
    num_generations             = NUM_GENERATIONS,
    gradient_accumulation_steps = 1,
    learning_rate               = 1e-5,
    max_completion_length       = 200,
    temperature                 = 0.7,
    beta                        = 0.1,   # KL penalty â€” prevents entropy collapse (fixed Run 2)
    logging_steps               = 1,
    save_steps                  = 9999,
    report_to                   = 'none',
    remove_unused_columns       = False,
)
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [env_reward_fn, format_reward_fn],
    args             = config,
    train_dataset    = dataset,
)

print(f'Training: {len(dataset)} prompts | {NUM_EPOCHS} epochs | {NUM_GENERATIONS} gen/prompt')
print(f'Estimated env calls: ~{len(dataset) * NUM_EPOCHS * NUM_GENERATIONS:,}')
result = trainer.train()
print(f'Done. Training loss: {result.training_loss:.4f}')
METRICS.summary()


In [ ]:
model.eval()
post_results = {}
print('=== POST-TRAINING EVAL ===')
for task_id in EVAL_TASKS:
    scores = []
    for seed in EVAL_SEEDS:
        score, raw, dec, fmt_ok = eval_one(model, task_id, seed)
        scores.append(score)
        print(f'  {task_id:<38} seed={seed}  score={score:.3f}  {dec}')
    post_results[task_id] = round(sum(scores) / len(scores), 4)

overall_post = sum(post_results.values()) / len(post_results)
print(f'\nOverall post-training mean: {overall_post:.3f}')

if baseline_results:
    print('\n=== DELTA (Before -> After) ===')
    for tid in EVAL_TASKS:
        b = baseline_results.get(tid, {}).get('mean', 0.0)
        p = post_results.get(tid, 0.0)
        print(f'  {tid:<38} {b:.3f} -> {p:.3f}  ({p-b:+.3f})')
else:
    print('\n[NOTE] baseline_results empty â€” run Part A first for before/after delta')


In [ ]:
# â”€â”€ 4-panel results figure (same layout as HF Space) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig = dark_fig(figsize=(16, 10))
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.30)

# Panel 1: Before / After eval bars
ax1   = fig.add_subplot(gs[0, 0])
tasks = list(EVAL_TASKS)
short = [t.replace('easy_','').replace('medium_','').replace('hard_','').replace('_',' ').title()
         for t in tasks]
xp    = np.arange(len(tasks))
b_vals = [baseline_results.get(t, {}).get('mean', 0.0) for t in tasks]
p_vals = [post_results.get(t, 0.0) for t in tasks]
ax1.bar(xp-0.2, b_vals, 0.35, label='Before GRPO', color='#f85149', alpha=0.85)
ax1.bar(xp+0.2, p_vals, 0.35, label='After GRPO',  color='#3fb950', alpha=0.85)
ax1.set_xticks(xp); ax1.set_xticklabels(short, rotation=35, ha='right', fontsize=7)
ax1.set_ylim(0, 1.05); ax1.axhline(0.5, color='#484f58', linestyle='--', alpha=0.6)
ax1.legend(fontsize=8, facecolor=PANEL, edgecolor='#30363d', labelcolor='#c9d1d9')
style_ax(ax1, f'Before vs After â€” {NUM_EPOCHS} Epochs GRPO')

# Panel 2: Per-task training mean
ax2 = fig.add_subplot(gs[0, 1])
task_means = {t:round(sum(v)/len(v),3) for t,v in METRICS.reward_by_task.items()}
if task_means:
    tm_tasks  = list(task_means.keys())
    tm_scores = list(task_means.values())
    tm_short  = [t.replace('easy_','').replace('medium_','').replace('hard_','').replace('_',' ').title()
                 for t in tm_tasks]
    colors    = ['#3fb950' if s>=0.7 else '#d29922' if s>=0.4 else '#f85149' for s in tm_scores]
    yp2 = range(len(tm_tasks))
    ax2.barh(list(yp2), tm_scores, color=colors, alpha=0.85, edgecolor=BG)
    ax2.set_yticks(list(yp2)); ax2.set_yticklabels(tm_short, fontsize=7)
    ax2.set_xlim(0, 1.05)
    ax2.axvline(0.7, color='#3fb950', linestyle='--', linewidth=1, alpha=0.5)
    for i,s in enumerate(tm_scores):
        ax2.text(s+0.01, i, f'{s:.2f}', va='center', color='#c9d1d9', fontsize=7)
style_ax(ax2, 'Per-Task Training Mean (all seeds)')

# Panel 3: Decision distribution pie
ax3 = fig.add_subplot(gs[1, 0])
dc  = dict(METRICS.decision_counts)
if dc:
    colors3 = ['#3fb950','#f85149','#d29922','#a371f7','#58a6ff','#39d353']
    wedges, _, autos = ax3.pie(list(dc.values()), labels=None,
                               autopct='%1.0f%%', colors=colors3[:len(dc)],
                               startangle=90, pctdistance=0.75,
                               wedgeprops=dict(edgecolor=BG, linewidth=1.5))
    for at in autos: at.set_color(BG); at.set_fontsize(8); at.set_fontweight('bold')
    ax3.legend(list(dc.keys()), loc='lower center', bbox_to_anchor=(0.5,-0.15),
               ncol=3, fontsize=7, facecolor=PANEL, edgecolor='#30363d', labelcolor='#c9d1d9')
ax3.set_facecolor(PANEL)
ax3.set_title('Decision Distribution', color=TXT, fontsize=10, fontweight='bold', pad=8)

# Panel 4: Reward curve
ax4 = fig.add_subplot(gs[1, 1])
if METRICS.reward_history:
    steps   = [s for s,_ in METRICS.reward_history]
    rewards = [r for _,r in METRICS.reward_history]
    ax4.plot(steps, rewards, color='#58a6ff', alpha=0.30, linewidth=1)
    if len(rewards) >= 5:
        w  = max(3, len(rewards)//15)
        sm = np.convolve(rewards, np.ones(w)/w, mode='valid')
        ax4.plot(steps[w-1:], sm, color='#79c0ff', linewidth=2, label=f'Smooth w={w}')
    mean_r = METRICS.recent_mean()
    ax4.axhline(mean_r, color='#f78166', linestyle='--', linewidth=1,
                label=f'Recent mean: {mean_r:.3f}')
    ax4.set_ylim(0, 1.05)
    ax4.legend(fontsize=7, facecolor=PANEL, edgecolor='#30363d', labelcolor='#c9d1d9')
style_ax(ax4, 'Reward Curve', xlabel='Training Step')

model_short = MODEL_NAME.split('/')[-1]
fig.suptitle(
    f'AP Commander GRPO â€” {model_short} | {NUM_EPOCHS}ep | {NUM_GENERATIONS}gen | '
    f'fmt={METRICS.fmt_rate():.1%} | parse_fails={METRICS.parse_failures} | '
    f'{datetime.datetime.now().strftime("%Y-%m-%d")}',
    color=TXT, fontsize=9, y=0.99
)
plt.savefig('/tmp/results.png', dpi=130, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: /tmp/results.png')

In [ ]:
# Standalone reward curve (for quick progress checks mid-training too)
if METRICS.reward_history:
    fig2 = dark_fig(figsize=(12, 4))
    ax   = fig2.add_subplot(111)
    steps   = [s for s,_ in METRICS.reward_history]
    rewards = [r for _,r in METRICS.reward_history]
    ax.plot(steps, rewards, color='#58a6ff', alpha=0.35, linewidth=1, label='Per-step')
    if len(rewards) >= 5:
        w  = max(3, len(rewards)//10)
        sm = np.convolve(rewards, np.ones(w)/w, mode='valid')
        ax.plot(steps[w-1:], sm, color='#79c0ff', linewidth=2.5, label=f'Smooth w={w}')
    ax.axhline(METRICS.recent_mean(), color='#f78166', linestyle='--',
               label=f'Recent mean: {METRICS.recent_mean():.3f}')
    ax.set_ylim(0, 1.05)
    ax.annotate(f'step {steps[-1]}  r={rewards[-1]:.3f}',
                xy=(steps[-1], rewards[-1]), xytext=(-40,12), textcoords='offset points',
                color='#f0f6fc', fontsize=8,
                arrowprops=dict(arrowstyle='->', color='#58a6ff', lw=1))
    ax.legend(fontsize=8, facecolor=PANEL, edgecolor='#30363d', labelcolor='#c9d1d9')
    style_ax(ax, 'Reward Curve', xlabel='Training Step', ylabel='Mean Batch Reward')
    fig2.suptitle(f'AP Commander GRPO â€” {MODEL_NAME.split("/")[-1]}', color=TXT, fontsize=9)
    plt.tight_layout()
    plt.savefig('/tmp/reward_curve.png', dpi=130, bbox_inches='tight', facecolor=BG)
    plt.show()
    print('Saved: /tmp/reward_curve.png')

In [ ]:
import shutil

model_slug = MODEL_NAME.split('/')[-1].lower().replace('.', '-')
ts         = datetime.datetime.now().strftime('%Y-%m-%d_%H%M')
run_dir    = f'/tmp/runs/grpo/{model_slug}-{NUM_EPOCHS}ep-{ts}'
os.makedirs(run_dir, exist_ok=True)

# Save LoRA adapters locally (do NOT merge with 4-bit base â€” load with PeftModel.from_pretrained)
adapter_dir = os.path.join(run_dir, 'adapter')
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapters saved to {adapter_dir}')

# Save plots and JSON
for src, dst in [
    ('/tmp/results.png',      'results.png'),
    ('/tmp/reward_curve.png', 'reward_curve.png'),
]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(run_dir, dst))

output = {
    'timestamp':       datetime.datetime.now().isoformat(),
    'run_dir':         run_dir,
    'model':           MODEL_NAME,
    'epochs':          NUM_EPOCHS,
    'num_generations': NUM_GENERATIONS,
    'eval_seeds':      EVAL_SEEDS,
    'eval_tasks':      EVAL_TASKS,
    'hardware':        'GPU (Colab/HF Spaces)',
    'baseline':        {t: baseline_results[t]['mean'] for t in baseline_results} if baseline_results else {},
    'post_training':   post_results,
    'delta':           {t: round(post_results.get(t,0) - (baseline_results[t]['mean'] if t in baseline_results else 0), 4)
                        for t in EVAL_TASKS},
    'metrics': {
        'total_reward_calls': METRICS.total_calls,
        'parse_failures':     METRICS.parse_failures,
        'format_rate':        round(METRICS.fmt_rate(), 4),
        'recent_mean':        round(METRICS.recent_mean(), 4),
        'decision_counts':    dict(METRICS.decision_counts),
        'per_task_mean':      {t: round(sum(v)/len(v),4) for t,v in METRICS.reward_by_task.items()},
    },
}
json_path = os.path.join(run_dir, 'training_results.json')
with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'JSON saved: {json_path}')

# â”€â”€ Download all artifacts from Colab â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import files
for fname in ['results.png', 'reward_curve.png', 'training_results.json']:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        files.download(fpath)

# â”€â”€ Optional: persist adapter to Google Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copytree(run_dir, f'/content/drive/MyDrive/ap_commander/{os.path.basename(run_dir)}')

print(f'\nAll done. Artifacts in {run_dir}')
print('Adapter can be loaded with: PeftModel.from_pretrained(base_model, adapter_dir)')

## Summary

| Artifact | Location |
|---|---|
| Baseline results | `runs/baselines/MODEL-DATETIME/baseline_results.json` |
| Baseline plot | `runs/baselines/MODEL-DATETIME/baseline_plot.png` |
| GRPO results | `runs/grpo/MODEL-NEP-DATETIME/training_results.json` |
| GRPO 4-panel plot | `runs/grpo/MODEL-NEP-DATETIME/results.png` |
| Reward curve | `runs/grpo/MODEL-NEP-DATETIME/reward_curve.png` |
| LoRA adapter | `runs/grpo/MODEL-NEP-DATETIME/adapter/` (local) |

All folders are timestamped â€” re-running never overwrites a previous run.

**Storage options** (Colab `/tmp` is lost when session ends):
- `files.download()` at the end of `save_grpo` auto-downloads artifacts to your machine
- Uncomment the Google Drive block to persist the full run folder across sessions